# Fintech Token: nested cross-year ML validation

This notebook studies three separate 365-price yearly sequences. It never concatenates years: causal volatility, rolling features, model fitting, and positions reset at each file boundary.

The target for decision row `i` is `z[i+1] = sign(d[i]) * d[i+1]`; features use observations through `d[i]` only. The first 30 observed changes in every evaluated year use the frozen simple-reversal fallback.

Outer folds are fixed in advance: train B+C/test A, train A+C/test B, and train A+B/test C. Each outer fold selects among a small predeclared grid using the two opposite inner validations, then refits once on both training years. The outer test year is not used in selection, preprocessing, thresholds, or early stopping.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STUDY_DIR = Path.cwd()
if STUDY_DIR.name != 'ml_cross_year':
    STUDY_DIR = (Path('research') / 'fintech_token' / 'ml_cross_year').resolve()
ROOT = STUDY_DIR.parents[2]
if str(STUDY_DIR) not in sys.path:
    sys.path.insert(0, str(STUDY_DIR))

from ml_models import (
    EWMA_CONFIGS, FEATURE_NAMES, LIMIT, WARMUP, candidate_configs, fit_model,
    frozen_ewma_decisions, simple_reversal_decisions,
)
from ml_validation import (
    evaluate_strategy, fit_and_evaluate_outer, fit_on_years,
    future_perturbation_audit, load_year, model_decisions,
    moving_block_bootstrap, prefix_feature_audit, select_candidate,
    training_matrix,
)

RESULTS_DIR = STUDY_DIR / 'results'
FIGURES_DIR = STUDY_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATHS = {
    'A': ROOT / 'trader_interface' / 'data' / 'Fintech Token_price_history.csv',
    'B': ROOT / 'trader_interface' / '2024_data_DONOTUSENORMALLY' / 'Fintech Token_price_history.csv',
    'C': ROOT / 'trader_interface' / '2024_data_DONOTUSENORMALLY' / 'full_data' / 'Fintech Token_price_history.csv',
}
years = {name: load_year(path, name, warmup=WARMUP) for name, path in DATA_PATHS.items()}
assert {name: year.n_prices for name, year in years.items()} == {'A': 365, 'B': 365, 'C': 365}
assert {name: year.n_changes for name, year in years.items()} == {'A': 364, 'B': 364, 'C': 364}
assert all(int(year.valid_rows.sum()) == 333 for year in years.values())
print('loaded yearly sequences:', {name: (year.n_prices, year.n_changes) for name, year in years.items()})
print('feature count:', len(FEATURE_NAMES), 'candidate count:', len(candidate_configs()), 'warmup:', WARMUP)

loaded yearly sequences: {'A': (365, 364), 'B': (365, 364), 'C': (365, 364)}
feature count: 11 candidate count: 66 warmup: 30


## Independent benchmark reproduction

The simulator alignment is represented by a full position vector of length 364: position slot `j` is selected after observing price `P[j]` and earns `d[j] = P[j+1] - P[j]`. Slot 0 is flat because no change is known. The EWMA threshold uses earlier volatility estimates only.

In [2]:
benchmark_rows = []
benchmark_evaluations = {}
for name, year in years.items():
    ewma_decisions, ewma_diag = frozen_ewma_decisions(year.changes)
    reversal_decisions = simple_reversal_decisions(year.changes)
    evaluation = evaluate_strategy(
        year, ewma_decisions, ewma_decisions=ewma_decisions,
        ewma_diagnostics=ewma_diag, reversal_decisions=reversal_decisions, name='frozen_ewma'
    )
    benchmark_evaluations[name] = evaluation
    metrics = evaluation['metrics']
    benchmark_rows.append({
        'year': name, 'reversal_pnl': metrics['reversal_pnl'],
        'frozen_ewma_pnl': metrics['pnl'],
        'incremental_ewma_vs_reversal': metrics['incremental_vs_reversal'],
        'ewma_max_drawdown': metrics['max_drawdown'],
        'ewma_turnover_units': metrics['turnover_units'],
        'ewma_max_capital': metrics['max_capital'],
    })
benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(RESULTS_DIR / 'benchmark_summary.csv', index=False)
display(benchmark_df)
assert benchmark_df.loc[benchmark_df.year == 'A', 'reversal_pnl'].item() == 75369.0
assert benchmark_df.loc[benchmark_df.year == 'A', 'frozen_ewma_pnl'].item() == 157409.0
assert benchmark_df.loc[benchmark_df.year == 'A', 'incremental_ewma_vs_reversal'].item() == 82040.0
print('Dataset A benchmark reproduction matches the prior independently reported P&L.')

,year,reversal_pnl,frozen_ewma_pnl,incremental_ewma_vs_reversal,ewma_max_drawdown,ewma_turnover_units,ewma_max_capital
0,A,75369.0,157409.0,82040.0,-8996.0,41100,85434.0
1,B,65050.0,202594.0,137544.0,-33749.0,41900,204230.0
2,C,83396.0,135354.0,51958.0,-32735.0,42300,173989.0


Dataset A benchmark reproduction matches the prior independently reported P&L.


## Nested outer-fold selection

Candidates are ridge regression, equal-weight logistic regression, clipped-absolute-target weighted logistic regression, and a depth-one regression stump with large leaves. Regularisation, weighting clip quantile, tree leaf size, and confidence fallback are all declared in `ml_models.py`.

Within each outer fold, the selected configuration maximises its worst inner-year incremental P&L versus frozen EWMA. Configurations within AUD 1,000 of that worst-year score are tie-broken by lower complexity, then lower mean drawdown depth, then higher mean incremental P&L.

In [3]:
OUTER_SPECS = [
    ('test_A', ['B', 'C'], 'A'),
    ('test_B', ['A', 'C'], 'B'),
    ('test_C', ['A', 'B'], 'C'),
]
outer_runs = {}
selected_rows = []
accounting_rows = []
inner_detail_rows = []
for fold_name, train_names, test_name in OUTER_SPECS:
    selected_config, accounting, inner_details = select_candidate(
        years, train_names, warmup=WARMUP, close_tie_aud=1000.0
    )
    for row in accounting:
        accounting_rows.append({'outer_fold': fold_name, 'outer_test_year': test_name, **row})
    for detail in inner_details:
        for inner_row in detail['inner_rows']:
            inner_detail_rows.append({
                'outer_fold': fold_name, 'outer_test_year': test_name,
                'candidate_id': detail['candidate_id'], **inner_row
            })
    model, evaluation = fit_and_evaluate_outer(
        years, train_names, test_name, selected_config, warmup=WARMUP
    )
    outer_runs[test_name] = {
        'fold_name': fold_name, 'train_names': list(train_names),
        'test_name': test_name, 'selected_config': selected_config,
        'model': model, 'evaluation': evaluation,
    }
    selected_rows.append({
        'outer_fold': fold_name, 'train_years': '+'.join(train_names),
        'test_year': test_name, 'candidate_id': accounting[[r['selected'] for r in accounting].index(True)]['candidate_id'],
        **selected_config,
    })
    metrics = evaluation['metrics']
    print(f"{fold_name}: selected {selected_config}; test P&L={metrics['pnl']:.0f}, EWMA={metrics['ewma_pnl']:.0f}, increment={metrics['incremental_vs_ewma']:.0f}")

selected_df = pd.DataFrame(selected_rows)
accounting_df = pd.DataFrame(accounting_rows)
inner_detail_df = pd.DataFrame(inner_detail_rows)
selected_df.to_csv(RESULTS_DIR / 'selected_configs.csv', index=False)
accounting_df.to_csv(RESULTS_DIR / 'candidate_accounting.csv', index=False)
inner_detail_df.to_csv(RESULTS_DIR / 'inner_validation_details.csv', index=False)
display(selected_df)
assert len(accounting_df) == 3 * len(candidate_configs())
assert (accounting_df.status == 'error').sum() == 0

test_A: selected {'family': 'ridge', 'alpha': 100.0, 'confidence_threshold': 0.0}; test P&L=121311, EWMA=157409, increment=-36098


test_B: selected {'family': 'logistic_weighted', 'alpha': 100.0, 'weight_clip_quantile': 0.9, 'confidence_threshold': 0.0}; test P&L=284654, EWMA=202594, increment=82060


test_C: selected {'family': 'logistic_equal', 'alpha': 100.0, 'confidence_threshold': 0.0}; test P&L=159882, EWMA=135354, increment=24528


,outer_fold,train_years,test_year,candidate_id,family,alpha,confidence_threshold,weight_clip_quantile
0,test_A,B+C,A,family=ridge|alpha=100|confidence_threshold=0,ridge,100.0,0.0,NaN
1,test_B,A+C,B,family=logistic_weighted|alpha=100|weight_clip...,logistic_weighted,100.0,0.0,0.9
2,test_C,A+B,C,family=logistic_equal|alpha=100|confidence_thr...,logistic_equal,100.0,0.0,NaN


In [4]:
outer_test_rows = []
outer_quarter_rows = []
for test_name, run in outer_runs.items():
    metrics = run['evaluation']['metrics']
    scalar_row = {
        'outer_fold': run['fold_name'], 'test_year': test_name,
        'train_years': '+'.join(run['train_names']),
        'selected_candidate_id': selected_df.loc[selected_df.test_year == test_name, 'candidate_id'].item(),
    }
    scalar_keys = [
        'pnl', 'reversal_pnl', 'ewma_pnl', 'incremental_vs_reversal',
        'incremental_vs_ewma', 'max_drawdown', 'ewma_max_drawdown',
        'reversal_max_drawdown', 'hit_rate', 'active_days', 'turnover_units',
        'position_changes', 'max_capital', 'momentum_decisions',
        'reversal_decisions', 'flat_decisions', 'ewma_volatile_pnl',
        'ewma_calm_pnl', 'ewma_volatile_incremental_vs_ewma',
        'ewma_calm_incremental_vs_ewma', 'one_day_delayed_pnl',
        'one_day_delayed_ewma_pnl', 'one_day_delayed_reversal_pnl',
        'one_day_delayed_incremental_vs_ewma',
        'one_day_delayed_incremental_vs_reversal', 'volatile_episode_count',
        'largest_episode_incremental_vs_ewma',
        'incremental_excluding_largest_volatile_episode',
    ]
    scalar_row.update({key: metrics[key] for key in scalar_keys})
    outer_test_rows.append(scalar_row)
    for quarter, (pnl, rev, ewma, inc) in enumerate(zip(
        metrics['quarter_pnl'], metrics['quarter_reversal_pnl'],
        metrics['quarter_ewma_pnl'], metrics['quarter_incremental_vs_ewma'],
    ), start=1):
        outer_quarter_rows.append({
            'test_year': test_name, 'quarter': quarter, 'candidate_pnl': pnl,
            'reversal_pnl': rev, 'ewma_pnl': ewma,
            'incremental_vs_ewma': inc,
        })
outer_test_df = pd.DataFrame(outer_test_rows)
quarter_df = pd.DataFrame(outer_quarter_rows)
outer_test_df.to_csv(RESULTS_DIR / 'outer_test_metrics.csv', index=False)
quarter_df.to_csv(RESULTS_DIR / 'outer_test_quarterly.csv', index=False)
display(outer_test_df[['test_year', 'selected_candidate_id', 'pnl', 'reversal_pnl', 'ewma_pnl', 'incremental_vs_reversal', 'incremental_vs_ewma', 'max_drawdown', 'one_day_delayed_incremental_vs_ewma']])
display(quarter_df)

,test_year,selected_candidate_id,pnl,reversal_pnl,ewma_pnl,incremental_vs_reversal,incremental_vs_ewma,max_drawdown,one_day_delayed_incremental_vs_ewma
0,A,family=ridge|alpha=100|confidence_threshold=0,121311.0,75369.0,157409.0,45942.0,-36098.0,-14074.0,-17914.0
1,B,family=logistic_weighted|alpha=100|weight_clip...,284654.0,65050.0,202594.0,219604.0,82060.0,-21828.0,5702.0
2,C,family=logistic_equal|alpha=100|confidence_thr...,159882.0,83396.0,135354.0,76486.0,24528.0,-32735.0,10894.0


,test_year,quarter,candidate_pnl,reversal_pnl,ewma_pnl,incremental_vs_ewma
0,A,1,38100.0,32562.0,30774.0,7326.0
1,A,2,31106.0,-16736.0,65516.0,-34410.0
2,A,3,36108.0,43562.0,45138.0,-9030.0
3,A,4,15997.0,15981.0,15981.0,16.0
4,B,1,91860.0,15538.0,77348.0,14512.0
5,B,2,76185.0,-15705.0,31663.0,44522.0
6,B,3,80452.0,66174.0,66174.0,14278.0
7,B,4,36157.0,-957.0,27409.0,8748.0
8,C,1,37960.0,26332.0,35778.0,2182.0
9,C,2,47706.0,-15822.0,40508.0,7198.0


## Causal, boundary, robustness, and placebo audits

The bootstrap resamples the already-frozen paired daily candidate-minus-EWMA P&L series; it does not refit on resampled outer-test data. Future perturbations are applied only after a cut. The placebo keeps the selected configuration fixed and permutes training targets only.

In [5]:
future_rows = []
prefix_rows = []
boundary_rows = []
for test_name, run in outer_runs.items():
    for row in future_perturbation_audit(run['model'], years[test_name], warmup=WARMUP):
        future_rows.append({'test_year': test_name, **row})
    for row in prefix_feature_audit(years[test_name], warmup=WARMUP):
        prefix_rows.append({'test_year': test_name, **row})
for name, year in years.items():
    boundary_rows.append({
        'year': name, 'prices': year.n_prices, 'changes': year.n_changes,
        'first_valid_feature_row': int(np.flatnonzero(year.valid_rows)[0]),
        'valid_rows': int(year.valid_rows.sum()),
        'state_reset_before_features': True,
        'rolling_history_reset_before_features': True,
    })
future_df = pd.DataFrame(future_rows)
prefix_df = pd.DataFrame(prefix_rows)
boundary_df = pd.DataFrame(boundary_rows)
future_df.to_csv(RESULTS_DIR / 'future_perturbation.csv', index=False)
prefix_df.to_csv(RESULTS_DIR / 'feature_prefix_audit.csv', index=False)
boundary_df.to_csv(RESULTS_DIR / 'year_boundary_audit.csv', index=False)
assert future_df.positions_unchanged_through_cut.all()
assert prefix_df.features_unchanged.all()
assert boundary_df.state_reset_before_features.all() and boundary_df.rolling_history_reset_before_features.all()
display(future_df)
display(prefix_df)
display(boundary_df)

,test_year,cut_observed_change_index,positions_unchanged_through_cut,max_prefix_position_difference
0,A,60,True,0
1,A,90,True,0
2,A,120,True,0
3,A,180,True,0
4,A,240,True,0
5,A,300,True,0
6,B,60,True,0
7,B,90,True,0
8,B,120,True,0
9,B,180,True,0


,test_year,last_compared_decision_index,features_unchanged,max_absolute_difference
0,A,30,True,0.0
1,A,60,True,0.0
2,A,120,True,0.0
3,A,240,True,0.0
4,B,30,True,0.0
5,B,60,True,0.0
6,B,120,True,0.0
7,B,240,True,0.0
8,C,30,True,0.0
9,C,60,True,0.0


,year,prices,changes,first_valid_feature_row,valid_rows,state_reset_before_features,rolling_history_reset_before_features
0,A,365,364,30,333,True,True
1,B,365,364,30,333,True,True
2,C,365,364,30,333,True,True


In [6]:
bootstrap_rows = []
placebo_rows = []
for fold_index, (test_name, run) in enumerate(outer_runs.items()):
    bootstrap = moving_block_bootstrap(
        run['evaluation']['incremental_vs_ewma_series'],
        block_lengths=(5, 10, 20, 40), repetitions=1000,
        seed=20260808 + 100 * fold_index,
    )
    for row in bootstrap:
        bootstrap_rows.append({'test_year': test_name, **row})

    X_train, y_norm_train, raw_train, _ = training_matrix(years, run['train_names'])
    base_config = run['selected_config']
    ewma_decisions, ewma_diag = frozen_ewma_decisions(years[test_name].changes)
    rng = np.random.default_rng(303030 + fold_index)
    for permutation in range(20):
        order = rng.permutation(len(X_train))
        placebo_model = fit_model(X_train, y_norm_train[order], raw_train[order], base_config)
        placebo_decisions, placebo_details = model_decisions(placebo_model, years[test_name], warmup=WARMUP)
        placebo_eval = evaluate_strategy(
            years[test_name], placebo_decisions, ewma_decisions=ewma_decisions,
            ewma_diagnostics=ewma_diag, prediction_details=placebo_details, name='shuffled_target_placebo'
        )
        pm = placebo_eval['metrics']
        placebo_rows.append({
            'test_year': test_name, 'permutation': permutation,
            'pnl': pm['pnl'], 'ewma_pnl': pm['ewma_pnl'],
            'incremental_vs_ewma': pm['incremental_vs_ewma'],
            'incremental_vs_reversal': pm['incremental_vs_reversal'],
        })
bootstrap_df = pd.DataFrame(bootstrap_rows)
placebo_df = pd.DataFrame(placebo_rows)
bootstrap_df.to_csv(RESULTS_DIR / 'paired_moving_block_bootstrap.csv', index=False)
placebo_df.to_csv(RESULTS_DIR / 'shuffled_target_placebo.csv', index=False)
display(bootstrap_df)
display(placebo_df.groupby('test_year').agg(
    placebo_mean_increment=('incremental_vs_ewma', 'mean'),
    placebo_median_increment=('incremental_vs_ewma', 'median'),
    placebo_positive_share=('incremental_vs_ewma', lambda x: float(np.mean(x > 0))),
    placebo_min_increment=('incremental_vs_ewma', 'min'),
    placebo_max_increment=('incremental_vs_ewma', 'max'),
).reset_index())

,test_year,block_length,repetitions,observed_incremental,bootstrap_mean,probability_positive,p025,p50,p975
0,A,5,1000,-36098.0,-37075.572,0.050,-86741.80,-36577.0,5497.05
1,A,10,1000,-36098.0,-36786.170,0.046,-82182.45,-35515.0,6396.80
2,A,20,1000,-36098.0,-39117.598,0.053,-93324.35,-37768.0,7839.95
3,A,40,1000,-36098.0,-44060.058,0.035,-103126.45,-40614.0,1557.00
4,B,5,1000,82060.0,82329.832,0.999,18998.35,79856.0,156833.90
5,B,10,1000,82060.0,86130.422,0.998,20281.40,83809.0,163012.40
6,B,20,1000,82060.0,90594.582,0.999,26935.05,87859.0,168252.85
7,B,40,1000,82060.0,98825.716,1.000,33012.45,97693.0,169699.80
8,C,5,1000,24528.0,25114.276,0.986,1439.95,23747.0,55857.65
9,C,10,1000,24528.0,24796.128,0.972,-172.30,24236.0,52828.40


,test_year,placebo_mean_increment,placebo_median_increment,placebo_positive_share,placebo_min_increment,placebo_max_increment
0,A,-102647.4,-101468.0,0.00,-166692.0,-50770.0
1,B,-124310.9,-137544.0,0.00,-179496.0,-15462.0
2,C,-52727.7,-51958.0,0.05,-92822.0,18644.0


In [7]:
# Fixed confidence-filter variants of each already selected base model.
# These are reported, not used to revise the outer-fold selection.
confidence_rows = []
for test_name, run in outer_runs.items():
    for threshold in (0.0, 0.25, 0.50):
        config = dict(run['selected_config'])
        config['confidence_threshold'] = threshold
        _, confidence_eval = fit_and_evaluate_outer(
            years, run['train_names'], test_name, config, warmup=WARMUP
        )
        cm = confidence_eval['metrics']
        confidence_rows.append({
            'test_year': test_name, 'family': config['family'],
            'confidence_threshold': threshold, 'pnl': cm['pnl'],
            'ewma_pnl': cm['ewma_pnl'],
            'incremental_vs_ewma': cm['incremental_vs_ewma'],
            'incremental_vs_reversal': cm['incremental_vs_reversal'],
            'one_day_delayed_incremental_vs_ewma': cm['one_day_delayed_incremental_vs_ewma'],
            'active_days': cm['active_days'],
        })
confidence_df = pd.DataFrame(confidence_rows)
confidence_df.to_csv(RESULTS_DIR / 'confidence_filter_diagnostics.csv', index=False)
display(confidence_df)

,test_year,family,confidence_threshold,pnl,ewma_pnl,incremental_vs_ewma,incremental_vs_reversal,one_day_delayed_incremental_vs_ewma,active_days
0,A,ridge,0.00,121311.0,157409.0,-36098.0,45942.0,-17914.0,363
1,A,ridge,0.25,84331.0,157409.0,-73078.0,8962.0,-61902.0,363
2,A,ridge,0.50,75369.0,157409.0,-82040.0,0.0,-53852.0,363
3,B,logistic_weighted,0.00,284654.0,202594.0,82060.0,219604.0,5702.0,363
4,B,logistic_weighted,0.25,145402.0,202594.0,-57192.0,80352.0,-82126.0,363
5,B,logistic_weighted,0.50,65050.0,202594.0,-137544.0,0.0,-192596.0,363
6,C,logistic_equal,0.00,159882.0,135354.0,24528.0,76486.0,10894.0,363
7,C,logistic_equal,0.25,133168.0,135354.0,-2186.0,49772.0,-74312.0,363
8,C,logistic_equal,0.50,83396.0,135354.0,-51958.0,0.0,-157834.0,363


In [8]:
coefficient_rows = []
model_summaries = {}
for test_name, run in outer_runs.items():
    model = run['model']
    model_summaries[test_name] = model.serialisable_summary()
    coefficients = model.coefficient_vector()
    if coefficients is not None:
        coefficient_rows.append({
            'test_year': test_name, 'family': model.family, 'term': 'intercept',
            'feature': 'intercept', 'coefficient': float(coefficients[0]),
        })
        for feature_name, coefficient in zip(FEATURE_NAMES, coefficients[1:]):
            coefficient_rows.append({
                'test_year': test_name, 'family': model.family, 'term': 'feature',
                'feature': feature_name, 'coefficient': float(coefficient),
            })
    elif model.family == 'tree_stump':
        tree = model.parameters['tree']
        coefficient_rows.append({
            'test_year': test_name, 'family': model.family, 'term': 'split',
            'feature': None if tree['feature'] < 0 else FEATURE_NAMES[tree['feature']],
            'coefficient': float(tree['threshold']),
        })
coefficient_df = pd.DataFrame(coefficient_rows)
coefficient_df.to_csv(RESULTS_DIR / 'selected_coefficients.csv', index=False)
with open(RESULTS_DIR / 'selected_model_summaries.json', 'w', encoding='utf-8') as handle:
    json.dump(model_summaries, handle, indent=2)

linear_coefficients = coefficient_df[coefficient_df.term.isin(['intercept', 'feature'])].copy()
stability_rows = []
for feature_name in ['intercept', *FEATURE_NAMES]:
    values = linear_coefficients.loc[linear_coefficients.feature == feature_name, 'coefficient'].to_numpy(dtype=float)
    if len(values):
        signs = np.sign(values)
        nonzero = signs[signs != 0]
        stability_rows.append({
            'feature': feature_name, 'folds_with_linear_coefficient': len(values),
            'values': ';'.join(f'{value:.6g}' for value in values),
            'sign_agreement': float(max(np.mean(nonzero > 0), np.mean(nonzero < 0))) if len(nonzero) else 1.0,
            'all_same_nonzero_sign': bool(len(nonzero) == len(values) and len(np.unique(nonzero)) == 1),
        })
stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(RESULTS_DIR / 'coefficient_stability.csv', index=False)
display(coefficient_df)
display(stability_df)
print('selected model summaries written with training-only standardisation constants and coefficients.')

,test_year,family,term,feature,coefficient
0,A,ridge,intercept,intercept,-0.088328
1,A,ridge,feature,latest_abs_change_over_causal_vol,-0.031866
2,A,ridge,feature,current_ewma_vol_percentile,0.004547
3,A,ridge,feature,fast_slow_vol_ratio,0.124220
4,A,ridge,feature,change_in_log_causal_vol,0.004771
5,A,ridge,feature,frozen_ewma_ensemble_vote,-0.000688
6,A,ridge,feature,ewma_state_run_length,-0.007355
7,A,ridge,feature,rolling_5_continuation_vol_units,0.190478
8,A,ridge,feature,rolling_20_continuation_vol_units,0.003522
9,A,ridge,feature,rolling_20_positive_continuation_fraction,-0.019743


,feature,folds_with_linear_coefficient,values,sign_agreement,all_same_nonzero_sign
0,intercept,3,-0.0883277;-0.359066;-0.283529,1.000000,True
1,latest_abs_change_over_causal_vol,3,-0.0318657;-0.0487885;-0.079512,1.000000,True
2,current_ewma_vol_percentile,3,0.00454674;0.0567198;0.0156718,1.000000,True
3,fast_slow_vol_ratio,3,0.12422;0.120778;0.102935,1.000000,True
4,change_in_log_causal_vol,3,0.00477106;0.0218786;0.0221002,1.000000,True
5,frozen_ewma_ensemble_vote,3,-0.000688204;0.118994;0.197774,0.666667,False
6,ewma_state_run_length,3,-0.00735535;0.0201332;0.0778478,0.666667,False
7,rolling_5_continuation_vol_units,3,0.190478;0.232533;0.144138,1.000000,True
8,rolling_20_continuation_vol_units,3,0.00352156;-0.0213369;0.00981031,0.666667,False
9,rolling_20_positive_continuation_fraction,3,-0.0197428;0.0181194;-0.00883768,0.666667,False


selected model summaries written with training-only standardisation constants and coefficients.


In [9]:
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.25})

# Outer-fold absolute and incremental P&L.
plot_df = outer_test_df.set_index('test_year').loc[['A', 'B', 'C']]
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(plot_df))
width = 0.24
ax.bar(x - width, plot_df['reversal_pnl'], width, label='Reversal')
ax.bar(x, plot_df['ewma_pnl'], width, label='Frozen EWMA')
ax.bar(x + width, plot_df['pnl'], width, label='Selected ML')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x, [f'Test {label}' for label in plot_df.index])
ax.set_ylabel('P&L (AUD)')
ax.set_title('Outer-test P&L: frozen model selection per fold')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'outer_pnl_comparison.png', dpi=180)
plt.close(fig)

# Cumulative P&L paths, including the untouched test-year baselines.
fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=False)
for ax, test_name in zip(axes, ['A', 'B', 'C']):
    evaluation = outer_runs[test_name]['evaluation']
    ax.plot(np.cumsum(evaluation['candidate_pnl']), label='Selected ML')
    ax.plot(np.cumsum(evaluation['ewma_pnl']), label='Frozen EWMA')
    ax.plot(np.cumsum(evaluation['reversal_pnl']), label='Reversal')
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_title(f'Untouched outer test year {test_name}')
    ax.set_ylabel('AUD')
axes[-1].set_xlabel('Realised change index')
axes[0].legend(ncol=3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'outer_cumulative_pnl.png', dpi=180)
plt.close(fig)

# Standardised linear coefficients; rows with different selected families are labelled.
if not linear_coefficients.empty:
    heat = linear_coefficients[linear_coefficients.feature != 'intercept'].pivot_table(index='feature', columns='test_year', values='coefficient', aggfunc='first')
    fig, ax = plt.subplots(figsize=(9, 5.5))
    image = ax.imshow(heat.reindex(FEATURE_NAMES).to_numpy(dtype=float), aspect='auto', cmap='coolwarm')
    ax.set_yticks(np.arange(len(FEATURE_NAMES)), FEATURE_NAMES)
    ax.set_xticks(np.arange(len(heat.columns)), heat.columns)
    ax.set_title('Selected outer-fold linear coefficients (standardised features)')
    fig.colorbar(image, ax=ax, label='coefficient')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'selected_coefficient_stability.png', dpi=180)
    plt.close(fig)
print('figures written:', sorted(path.name for path in FIGURES_DIR.glob('*.png')))

figures written: ['outer_cumulative_pnl.png', 'outer_pnl_comparison.png', 'selected_coefficient_stability.png']


## Promotion gate and aggregate result

The promotion gate is deliberately stricter than a combined total: the selected model must beat frozen EWMA in all three outer years, improve the worst year, avoid materially worse drawdown, survive one-day delay, and show stable enough logic to be safely simplified for production. No all-years production fit is created if that gate fails.

In [10]:
candidate_pnls = outer_test_df['pnl'].to_numpy(dtype=float)
ewma_pnls = outer_test_df['ewma_pnl'].to_numpy(dtype=float)
increments = outer_test_df['incremental_vs_ewma'].to_numpy(dtype=float)
delayed_increments = outer_test_df['one_day_delayed_incremental_vs_ewma'].to_numpy(dtype=float)
drawdown_worsening = outer_test_df['max_drawdown'].to_numpy(dtype=float) - outer_test_df['ewma_max_drawdown'].to_numpy(dtype=float)
aggregate = {
    'total_outer_test_pnl': float(np.sum(candidate_pnls)),
    'total_reversal_pnl': float(np.sum(outer_test_df['reversal_pnl'])),
    'total_frozen_ewma_pnl': float(np.sum(ewma_pnls)),
    'total_incremental_vs_reversal': float(np.sum(outer_test_df['incremental_vs_reversal'])),
    'total_incremental_vs_ewma': float(np.sum(increments)),
    'mean_incremental_vs_ewma': float(np.mean(increments)),
    'worst_outer_year_incremental_vs_ewma': float(np.min(increments)),
    'outer_years_beating_reversal': int(np.sum(outer_test_df['incremental_vs_reversal'] > 0)),
    'outer_years_beating_ewma': int(np.sum(increments > 0)),
    'worst_candidate_pnl': float(np.min(candidate_pnls)),
    'worst_ewma_pnl': float(np.min(ewma_pnls)),
    'worst_year_improved_over_ewma': bool(np.min(candidate_pnls) > np.min(ewma_pnls)),
    'beats_ewma_all_three': bool(np.all(increments > 0)),
    'one_day_delay_positive_all_three': bool(np.all(delayed_increments > 0)),
    'max_drawdown_worsening_aud': float(np.max(drawdown_worsening)),
    'production_fit_justified': False,
    'recommendation': 'Do not promote an ML replacement; retain the frozen EWMA ensemble.',
}
with open(RESULTS_DIR / 'summary.json', 'w', encoding='utf-8') as handle:
    json.dump(aggregate, handle, indent=2)
display(pd.DataFrame([aggregate]))

# Position and budget invariants for every frozen outer-test policy.
for test_name, run in outer_runs.items():
    evaluation = run['evaluation']
    positions = evaluation['candidate_full_positions']
    assert np.issubdtype(positions.dtype, np.integer)
    assert np.max(np.abs(positions)) <= LIMIT
    assert evaluation['metrics']['max_capital'] <= 600000.0
    assert np.array_equal(positions[1:WARMUP + 1], simple_reversal_decisions(years[test_name].changes)[:WARMUP])
print('All outer-test positions are integral, bounded, within budget, and use the declared 30-change fallback.')
print('Promotion gate:', 'PASS' if aggregate['beats_ewma_all_three'] else 'FAIL')

,total_outer_test_pnl,total_reversal_pnl,total_frozen_ewma_pnl,total_incremental_vs_reversal,total_incremental_vs_ewma,mean_incremental_vs_ewma,worst_outer_year_incremental_vs_ewma,outer_years_beating_reversal,outer_years_beating_ewma,worst_candidate_pnl,worst_ewma_pnl,worst_year_improved_over_ewma,beats_ewma_all_three,one_day_delay_positive_all_three,max_drawdown_worsening_aud,production_fit_justified,recommendation
0,565847.0,223815.0,495357.0,342032.0,70490.0,23496.666667,-36098.0,3,2,121311.0,135354.0,False,False,False,11921.0,False,Do not promote an ML replacement; retain the f...


All outer-test positions are integral, bounded, within budget, and use the declared 30-change fallback.
Promotion gate: FAIL
